# Train and evaluate the cartoon colorizer (Colab Pro / Colab+)

**Runtime → Change runtime type → A100 or L4 GPU** (T4 also works, just slower), then **Runtime → Run all**.

What this notebook does:
1. Trains the CycleGAN on six public-domain Max Fleischer *Superman* shorts (1941–43) from the Internet Archive, in mixed precision.
2. Trains an **ablation** with the identity loss turned off, using identical settings.
3. Scores both models and two baselines on frames from a **seventh episode the models never saw**, against the true colors: PSNR, SSIM, ΔE2000, LPIPS, FID and colorfulness, with 95% confidence intervals.
4. Benchmarks inference speed across batch sizes and precisions.
5. Saves everything to Google Drive and downloads a results zip.

**If Colab disconnects:** reconnect and choose Run all again. Checkpoints live on Drive, so training resumes from the last finished epoch.

In [ ]:
import os, torch
REPO_BRANCH = 'main'
if not os.path.exists('/content/Cartoon-Colorization-Using-CycleGAN'):
    !git clone -q -b $REPO_BRANCH https://github.com/ArunaSamhitha/Cartoon-Colorization-Using-CycleGAN.git /content/Cartoon-Colorization-Using-CycleGAN
else:
    !git -C /content/Cartoon-Colorization-Using-CycleGAN pull -q
%cd /content/Cartoon-Colorization-Using-CycleGAN
!pip install -q -r requirements.txt
GPU = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE'
print('GPU:', GPU if GPU != 'NONE' else 'NONE - Runtime > Change runtime type > A100, L4 or T4')

In [ ]:
# ---- Settings ----
# 'quick': 128px, ~1,500 frames. Good first run; both models fit in one session on an A100 or L4.
# 'full' : 256px, 9 residual blocks. Sharper results, roughly 6x the compute of 'quick'.
PRESET = 'quick'
RUN_ABLATION = True          # also train a model without the identity loss, for comparison
SAVE_TO_DRIVE = True         # keep checkpoints and results on Drive so a disconnect can resume

PRESETS = {
    'quick': dict(IMAGE_SIZE=128, N_BLOCKS=6, BATCH=8, EPOCHS=30, DECAY_START=15, FRAME_EVERY=48),
    'full':  dict(IMAGE_SIZE=256, N_BLOCKS=9, BATCH=4, EPOCHS=40, DECAY_START=20, FRAME_EVERY=48),
}
cfg = PRESETS[PRESET]
HELDOUT_EVERY = 24           # one held-out frame per second of the test episode (~400 frames)

ROOT = '/content/drive/MyDrive/cyclegan' if SAVE_TO_DRIVE else '/content/cyclegan_out'
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
RUNS = f'{ROOT}/runs_{PRESET}'
RESULTS = f'{ROOT}/results_{PRESET}'
os.makedirs(RUNS, exist_ok=True); os.makedirs(RESULTS, exist_ok=True)
print(PRESET, cfg, '->', ROOT)

## 1. Download the episodes and extract frames

In [ ]:
import urllib.parse, subprocess
os.makedirs('clips/train', exist_ok=True); os.makedirs('clips/test', exist_ok=True)
BASE = 'https://archive.org/download/fleischer-superman/'
TRAIN = ["Max Fleischer's Superman： The Mechanical Monsters (1941; HD).mp4",
         "Max Fleischer's Superman： Billion Dollar Limited (1942; HD).mp4",
         "Max Fleischer's Superman： The Arctic Giant (1942; HD).mp4",
         "Max Fleischer's Superman： The Bulleteers (1942; HD).mp4",
         "Max Fleischer's Superman： The Magnetic Telescope (1942; HD).mp4",
         "Max Fleischer's Superman： The Electric Earthquake (1942; HD).mp4"]
TEST = ["Max Fleischer's Superman： Volcano (1942; HD).mp4"]

for i, name in enumerate(TRAIN):
    if not os.path.exists(f'clips/train/{i}.mp4'):
        print('Downloading', name)
        subprocess.run(['wget', '-q', '-O', f'clips/train/{i}.mp4', BASE + urllib.parse.quote(name)], check=True)
if not os.path.exists('clips/test/0.mp4'):
    print('Downloading', TEST[0])
    subprocess.run(['wget', '-q', '-O', 'clips/test/0.mp4', BASE + urllib.parse.quote(TEST[0])], check=True)
# A 30-second black-and-white clip from the held-out episode, for the video demo
subprocess.run(['ffmpeg', '-y', '-loglevel', 'error', '-ss', '60', '-t', '30', '-i', 'clips/test/0.mp4',
                '-vf', 'format=gray', 'clips/test_bw_30s.mp4'], check=True)

if not os.path.exists('data/frames'):
    !python scripts/extract_frames.py --videos clips/train --out data/frames --every {cfg['FRAME_EVERY']}
if not os.path.exists('data/heldout'):
    !python scripts/extract_frames.py --videos clips/test/0.mp4 --out data/heldout --every {HELDOUT_EVERY}
N_TRAIN, N_HELDOUT = len(os.listdir('data/frames')), len(os.listdir('data/heldout'))
print(f'{N_TRAIN} training frames, {N_HELDOUT} held-out frames')

## 2. Train

Each epoch prints its losses, `seconds` and `img_per_s`. **First epoch's seconds × epochs ≈ total time** for that run, so check it after the first epoch if you're watching compute units.

In [ ]:
def train(name, extra=''):
    out = f'{RUNS}/{name}'
    last = f'{out}/checkpoints/last.pt'
    resume = f'--resume "{last}"' if os.path.exists(last) else ''
    if resume:
        done = torch.load(last, map_location='cpu', weights_only=False)['epoch'] + 1
        if done >= cfg['EPOCHS']:
            print(f'{name}: already trained ({done} epochs)'); return
        print(f'{name}: resuming after epoch {done}')
    flags = (f"--image-size {cfg['IMAGE_SIZE']} --n-blocks {cfg['N_BLOCKS']} --batch-size {cfg['BATCH']} "
             f"--epochs {cfg['EPOCHS']} --decay-start {cfg['DECAY_START']}")
    !python train.py --frames data/frames --out "{out}" --amp --workers 4 {flags} {extra} {resume}

train('full')

In [ ]:
if RUN_ABLATION:
    train('no_identity', '--lambda-identity 0')

## 3. Watch progress

In [ ]:
import glob
from IPython.display import Image, Markdown, display
last_sample = sorted(glob.glob(f'{RUNS}/full/samples/*.png'))[-1]
print(last_sample, '(rows: grayscale input | colorized | reconstructed)'); display(Image(last_sample))

## 4. Evaluate on the held-out episode

Every held-out frame still has its true colors, so each output is scored against ground truth. `grayscale` (no color) and `mean_tint` (the training set's average color laid over the frame) are the baselines a real colorizer has to beat.

In [ ]:
models = f'--model full="{RUNS}/full/checkpoints/colorizer.pt"'
if RUN_ABLATION:
    models += f' --model no_identity="{RUNS}/no_identity/checkpoints/colorizer.pt"'
!python scripts/evaluate.py --frames data/heldout --train-frames data/frames {models} --width 256 --out "{RESULTS}/eval"
display(Markdown(open(f'{RESULTS}/eval/results.md').read()))

## 5. Inference speed, loss curves, results grid and a colorized video

In [ ]:
!python scripts/benchmark.py --ckpt "{RUNS}/full/checkpoints/colorizer.pt" --out "{RESULTS}/benchmark.json"
runs = f'--run full="{RUNS}/full"' + (f' --run no_identity="{RUNS}/no_identity"' if RUN_ABLATION else '')
!python scripts/plot_history.py {runs} --out assets/loss_curves.png
!python scripts/make_results.py --ckpt "{RUNS}/full/checkpoints/colorizer.pt" --frames data/heldout --out assets/results.png --n 4
!python colorize.py --ckpt "{RUNS}/full/checkpoints/colorizer.pt" --input clips/test_bw_30s.mp4 --output "{RESULTS}/test_colorized.mp4"
display(Image('assets/loss_curves.png')); display(Image('assets/results.png'))

## 6. Summary and downloads

In [ ]:
import json, shutil
def history(name):
    rows = {}
    for line in open(f'{RUNS}/{name}/history.json'):
        if line.strip():
            r = json.loads(line); rows[r['epoch']] = r
    return [rows[k] for k in sorted(rows)]

h = history('full')
summary = {
    'gpu': GPU, 'preset': PRESET, **cfg,
    'train_frames': N_TRAIN, 'heldout_frames': N_HELDOUT,
    'epochs_trained': len(h),
    'train_minutes_full': round(sum(r['seconds'] for r in h) / 60, 1),
    'median_train_img_per_s': sorted(r['img_per_s'] for r in h)[len(h) // 2],
    'peak_train_mem_gb': max(r.get('peak_mem_gb', 0) for r in h),
    'eval': json.load(open(f'{RESULTS}/eval/metrics.json'))['summary'],
    'benchmark_best': json.load(open(f'{RESULTS}/benchmark.json'))['best'],
}
if RUN_ABLATION:
    summary['train_minutes_no_identity'] = round(sum(r['seconds'] for r in history('no_identity')) / 60, 1)
json.dump(summary, open(f'{RESULTS}/summary.json', 'w'), indent=2)
for f in ('assets/results.png', 'assets/loss_curves.png'):
    shutil.copy(f, RESULTS)
for name in ['full'] + (['no_identity'] if RUN_ABLATION else []):
    shutil.copy(f'{RUNS}/{name}/history.json', f'{RESULTS}/history_{name}.json')
print(json.dumps(summary, indent=2))

shutil.make_archive('/content/cyclegan_results', 'zip', RESULTS)
from google.colab import files
files.download('/content/cyclegan_results.zip')